**Task 1. Imports and loading**

Import the data and packages that you've learned are needed for building regression models.

In [ ]:
# Import packages for data manipulation
import pandas as pd
import numpy as np


# Import packages for data visualization
import matplotlib.pyplot as plt
import seaborn as sns 


# Import packages for data preprocessing
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
import sklearn.metrics as matrics
from sklearn.metrics import classification_report


# Import packages for data modeling

from sklearn.model_selection import train_test_split

In [ ]:
Load the TikTok dataset.
# Load dataset into dataframe
data = pd.read_csv("tiktok_dataset.csv")

** Explore data with EDA**


Analyze the data and check for and handle missing values and duplicates

In [ ]:
# Display first few rows
data.head()
# Get number of rows and columns
data.shape
# Get data types of columns
data.dtypes

In [ ]:
# Get basic information
data.info()

In [ ]:
# Generate basic descriptive stats
data.describe()

Check for and handle missing values.

In [ ]:
# Check for missing values
data.isna().sum()

In [ ]:
# Drop rows with missing values
data = data.dropna(axis=0).reset_index(drop=True)

In [ ]:
# Display first few rows after handling missing values
data.isna().sum()

Check for and handle duplicates.

In [ ]:
# Check for duplicates
data.duplicated().sum()

It seems there is no duplication.

In [ ]:
# Create a boxplot to visualize distribution of `video_duration_sec`
plt.figure(figsize=(6,2))
sns.boxplot(x = data['video_duration_sec'])

In [ ]:
# Create a boxplot to visualize distribution of `video_view_count`
plt.figure(figsize=(6,2))
sns.boxplot(x = data['video_view_count'])

In [ ]:
# Create a boxplot to visualize distribution of `video_like_count`
plt.figure(figsize=(6,2))
sns.boxplot(x=data['video_like_count'])

In [ ]:
# Create a boxplot to visualize distribution of `video_comment_count`
plt.figure(figsize=(6,2))
sns.boxplot(x=data['video_comment_count'])

In [ ]:
# Check for and handle outliers for video_like_count
Q1 = data['video_like_count'].quantile(0.25)
Q3 = data['video_like_count'].quantile(0.75)

IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR

lower_limit = Q1 - 1.5 * IQR

data.loc[data['video_like_count'] > upper_limit, 'video_like_count'] = upper_limit
plt.figure(figsize=(6,2))
sns.boxplot(x=data['video_like_count'], data=data)

Check class balance of the target variable. Remember, the goal is to predict whether the user of a given post is verified or unverified

In [ ]:
# Check class balance
c = data['verified_status'].value_counts()
count_verified = 1200/(1200 + 17884)
count_unverified = 17884/(1200 + 17884)
print(c)
print(f"verified count: {count_verified}")
print(f"unverified count: {count_unverified}")

Approximately 93.7% of the dataset represents videos posted by unverified accounts and 6.2% represents videos posted by verified accounts. So the outcome variable is not very balanced.

Use resampling to create class balance in the outcome variable, if needed.

In [ ]:
# Use resampling to create class balance in the outcome variable, if needed

# Identify data points from majority and minority classes
from sklearn.utils import resample
majority = data[data['verified_status'] == 'not verified']
minority = data[data['verified_status'] == 'verified']


# Upsample the minority class (which is "verified")

minority_upsampled = resample(minority, replace =True, n_samples=len(majority), random_state=0)

# Combine majority class with upsampled minority class
data_upsampled = pd.concat([majority, minority_upsampled]).reset_index(drop=True)

# Display new class counts
data_upsampled['verified_status'].value_counts()

In [ ]:
# Get the average `video_transcription_text` length for claims and the average `video_transcription_text` length for opinions
data_upsampled.groupby('verified_status')['video_transcription_text'].apply(lambda x: x.str.len().mean())

Extract the length of each video_transcription_text and add this as a column to the dataframe, so that it can be used as a potential feature in the model.

In [ ]:
# Extract the length of each `video_transcription_text` and add this as a column to the dataframe
data_upsampled['text_length'] = data_upsampled['video_transcription_text'].str.len()

In [ ]:
# Display first few rows of dataframe after adding new column
data_upsampled.head()

Visualize the distribution of video_transcription_text length for videos posted by verified accounts and videos posted by unverified accounts.

In [ ]:
# Visualize the distribution of `video_transcription_text` length for videos posted by verified accounts and videos posted by unverified accounts
# Create two histograms in one plot
sns.histplot(x='text_length', hue='verified_status', data=data_upsampled, multiple='stack')
plt.title('Distribution of Video Transcription Text length by Verification Status')
plt.show()


** Examine correlations**






Next, code a correlation matrix to help determine most correlated variables.

In [ ]:
# Code a correlation matrix to help determine most correlated variables
correlation_matrix = data_upsampled.corr(numeric_only=True)
correlation_matrix

Visualize a correlation heatmap of the data.

In [ ]:
# Create a heatmap to visualize how correlated variables are
plt.figure(figsize=(10,8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

** Select variables¶**
Set your Y and X variables.

Select the outcome variable.

In [ ]:
# Select outcome variable
y = data_upsampled['verified_status']

In [ ]:
# Select features
x = data_upsampled[['video_duration_sec', 'claim_status', 'author_ban_status', 'video_view_count', 'video_share_count', 'video_download_count', 'video_comment_count']]


# Display first few rows of features dataframe
print(x)

**Train-test split**


Split the data into training and testing sets.

In [ ]:
# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.03, random_state=25)

Confirm that the dimensions of the training and testing sets are in alignment.

In [ ]:
# Get shape of each training and testing set
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

**Encode variables**


Check the data types of the features.

In [ ]:
# Check data types

print(x.dtypes)


In [ ]:
# Get unique values in `claim_status`


x_train["claim_status"].unique

In [ ]:
# Get unique values in `author_ban_status`

x_train['author_ban_status'].unique

As shown above, the claim_status and author_ban_status features are each of data type object currently. In order to work with the implementations of models through sklearn, these categorical features will need to be made numeric. One way to do this is through one-hot encoding.

Encode categorical features in the training set using an appropriate method.

In [ ]:
# Select the training features that needs to be encoded
features = x_train[['claim_status', 'author_ban_status']]

# Display first few rows
x_train[['claim_status', 'author_ban_status']].head()

In [ ]:
# Set up an encoder for one-hot encoding the categorical features
encoder = OneHotEncoder(drop='first', sparse_output=False)

In [ ]:
converted_columns = encoder.fit_transform(features)
print(converted_column)

In [ ]:
# Get feature names from encoder
names = encoder.get_feature_names_out()

In [ ]:
# Display first few rows of encoded training features
converted_columns

In [ ]:
# Place encoded training features (which is currently an array) into a dataframe
X_trian_encoder =  pd.DataFrame(data=converted_columns, columns=names)


# Display first few rows
X_trian_encoder.head()

In [ ]:
# Display first few rows of `X_train` with `claim_status` and `author_ban_status` columns dropped (since these features are being transformed to numeric)
x_train.drop(columns=["claim_status","author_ban_status"], inplace=True)

In [ ]:
#  Concatenate `X_train` and `X_train_encoded_df` to form the final dataframe for training data (`X_train_final`)



updated_xtrain = pd.concat([x_train.drop(columns=["claim_status", "author_ban_status"], errors='ignore').reset_index(drop=True), X_trian_encoder], axis=1)

# Display first few rows (changed to match the variable name created above)
updated_xtrain.head()

In [ ]:
# Check data type of outcome variable
updated_dataframe.dtypes

In [ ]:
# Get unique values of outcome variable
y.unique()

A shown above, the outcome variable is of data type object currently. One-hot encoding can be used to make this variable numeric.

Encode categorical values of the outcome variable the training set using an appropriate method.

In [ ]:
# Set up an encoder for one-hot encoding the categorical outcome variable
y_encoder = OneHotEncoder(drop='first', sparse_output=False)

In [ ]:
# Encode the training outcome variable
updated_ytrain = y_encoder.fit_transform(y_train.values.reshape(-1,1)).ravel()

# Display the encoded training outcome variable
y_train_encoded

**Model building¶**


Construct a model and fit it to the training set.

In [ ]:
# Construct a logistic regression model and fit it to the training set

clf = LogisticRegression(random_state=0, max_iter=800).fit(updated_xtrain, updated_ytrain=\][po[]]

**Results and evaluation**
Evaluate your model.


Encode categorical features in the testing set using an appropriate method.

In [ ]:
# Select the testing features that needs to be encoded
x_test_to_encoder= x_test[['claim_status', 'author_ban_status']]

# Display first few rows
x_test_to_encoder

In [ ]:
# Transform the testing features using the encoder
x_test_encoding = encoder.transform(x_test_to_encoder)


# Display first few rows of encoded testing features
x_test_encoding

In [ ]:
# Place encoded testing features (which is currently an array) into a dataframe
X_test_encoder = pd.DataFrame(data=x_test_encoding, columns=encoder.get_feature_names_out())


# Display first few rows

X_test_encoder.head()

In [ ]:
# Display first few rows of `X_test` with `claim_status` and `author_ban_status` columns dropped (since these features are being transformed to numeric)
x_test.head()

In [ ]:
 # Concatenate `X_test` and ` X_test_encoder` to form the final dataframe for training data (`X_test_final`)
updated_x_test = pd.concat([x_test.drop(columns=['claim_status', 'author_ban_status'], errors='ignore').reset_index(drop=True), X_test_encoder], axis=1)

# Display first few rows
updated_x_test

Test the logistic regression model. Use the model to make predictions on the encoded testing set.

In [ ]:
# Use the logistic regression model to get predictions on the encoded testing set
y_predict = clf.predict(updated_x_test)

In [ ]:
# Display the predictions on the encoded testing set
y_predict


In [ ]:
# Display the true labels of the testing set
y_test

Encode the true labels of the testing set so it can be compared to the predictions.

In [ ]:
# Encode the testing outcome variable
updated_y_test= y_encoder.transform(y_test.values.reshape(-1,1)).ravel()


# Display the encoded testing outcome variable
updated_y_test

Confirm again that the dimensions of the training and testing sets are in alignment since additional features were added.

In [ ]:
# Get shape of each training and testing set
updated_xtrain.shape, updated_ytrain.shape, updated_x_test.shape, updated_y_test.shape

Task 4b. Visualize model results






Create a confusion matrix to visualize the results of the logistic regression model.

In [ ]:
# Compute values for confusion matrix
cm = matrics.confusion_matrix(updated_y_test, y_predict, labels=clf.classes_)

# Create display of confusion matrix
cm_display = matrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)

# Plot confusion matrix
cm_display.plot()

# Display plot
plt.show()

Create a classification report that includes precision, recall, f1-score, and accuracy metrics to evaluate the performance of the logistic regression model.

In [ ]:
# Create a classification report
target_labels = ["verified", "unverified"]
print(classification_report(updated_y_test, y_predict, target_names=target_labels))

The classification report shows that the precision and recall for 'not verified' accounts are 61% and 86%, respectively, and the accuracy is 65%. Since 'not verified' status is the target that our model predict, we specifically focused on it. Still, verified status have it own precision and recall, and the weighted average is a combination of verified and unverified evaluation metrics.

### **Interpret model coefficients**

In [ ]:
# Get the feature names from the model and the model coefficients (which represent log-odds ratios)
# Place into a DataFrame for readability
pd.DataFrame(data={"Feature Name":clf.feature_names_in_, "Model Coefficient":clf.coef_[0]})


4d. Conclusion**

The main takeaways from this lab is the following:

- Data balance and multicollinearity should be checked before building the model, as they can lead to inaccurate model predictions. In this case, we dropped the video_like_count column because it had the highest correlation with other independent variables (video features), which could skew the results.

- The confusion matrix reflects that the model predicts 43.3% of the verified accounts that were actually verified and 21.8% were predicted as being not verified that are actually unverified. However, 27.8% were predicted as being unverified despite being verified and 7% were predicted as verified despite being unverified.

- The model's precision of 61% for 'unverified' accounts is less than ideal, but the recall of 86% is very good. The overall accuracy of 65% is lower than what is generally considered acceptable.
- 
- Based on the logistic regression model: for every additional second in the video duration, the log odds that the account is verified increase by 0.009.

This logistic regression model is built to predict the status of an account based on video features. The estimated model coefficients show that video_duration_sec has the highest correlation with the account status: longer videos are highly correlated with verified accounts. Other features have small coefficients, which means they have a small association with the account status.
